# 04 — Evaluation & Tuning
**Project:** Customer Churn Prediction (Telco) &nbsp;|&nbsp; **Builds on:** Milestones 01–03

**Objective:** evaluate the models with metrics that are actually appropriate for an imbalanced classification problem
(not accuracy alone), then tune each model's hyperparameters with `GridSearchCV` to get the best fair comparison before
picking a final model.

Every cell in this notebook computes real output from whatever data is loaded — including the interpretation paragraphs at
the end, which are generated from the actual numbers, not written in advance. Run the whole notebook top to bottom (or
**Run All**) and everything, including the write-up, fills itself in.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import cross_val_score, train_test_split

from src.config import RANDOM_STATE, TEST_SIZE
from src.data_preparation import clean, load_raw, make_xy
from src.preprocessing import build_logistic_class_weighted, build_random_forest_class_weighted
from src.tuning import PARAM_GRIDS, build_grid_search

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Setup complete. Random seed:", RANDOM_STATE)

## 1. Recap: load, clean, split
Same seed and logic as Milestones 01–03.

In [ ]:
raw = load_raw()
df, cleaning_log = clean(raw)
X, y = make_xy(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape, "| Churn rate (train):", round(y_train.mean(), 3))

## 2. Why accuracy is not the right metric here

In [ ]:
print(f"If a model predicted 'stays' for every customer, it would be right {(1 - y_train.mean()):.1%} of the time")
print("...while never once identifying an actual churner (recall = 0, precision = undefined).")
print("This is exactly what the Milestone 01 dummy classifier did — see reports/baseline_metrics.json.")

Accuracy rewards a model for being right on the *majority* class, which here is "stays." A model can score high on
accuracy while being completely useless at the one thing this project needs: finding likely churners. That is why this
notebook leads with `classification_report`, `confusion_matrix`, and ROC-AUC instead.

## 3. Baseline cross-validation (before tuning)
Using `cross_val_score` directly, on the two class-weighted models carried forward from Milestone 03, scored on ROC-AUC
(a threshold-independent metric appropriate for an imbalanced target).

In [ ]:
from sklearn.model_selection import cross_val_score

candidates = {
    "logistic_regression": build_logistic_class_weighted(X_train),
    "random_forest": build_random_forest_class_weighted(X_train),
}

baseline_cv = {}
for name, pipeline in candidates.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="roc_auc")
    baseline_cv[name] = scores
    print(f"{name:20s} ROC-AUC: {scores.mean():.4f} +/- {scores.std():.4f}  (folds: {np.round(scores, 3)})")